# LLaMEA Noisy BBOB Optimization Algorithm Evolution

This notebook acts as the main entry point to prototype, configure, and execute LLaMEA-driven evolution of optimization algorithms on BBOB functions. 

## Environment Configuration
Before running this notebook, ensure your packages are installed using `uv sync --all-extras`.

In [1]:
# Make sure the src directory is in the path
import sys
import os
sys.path.insert(0, os.path.abspath('../../src'))

import numpy as np
import plotly.graph_objects as go

from aad_llm.noisy_bbob import BBOBProblem
from aad_llm.prompts import TASK_PROMPT_TEMPLATE, EXAMPLE_PROMPT, FORMAT_PROMPT
import aad_llm.runner as runner


## 1. Sanity Check: Additive Gaussian Noise Wrapper
Let's test the `BBOBProblem` class on Problem 1 and plot the distribution of additive noise to confirm it works correctly.

In [2]:
# Config parameters
problem_id = 1
instance_id = 1
dim = 3
noise_std = 0.5  # Standard deviation of additive Gaussian noise

# Instantiate BBOBProblem
noisy_func = BBOBProblem(
    problem_id=problem_id,
    dim=dim,
    instance_id=instance_id,
    noise_std=noise_std
)
true_opt = noisy_func.true_optimum

print(f"Loaded Problem {problem_id} (Instance {instance_id}, DIM={dim})")
print(f"True target optimum (minimum): {true_opt:.4f}")

# Evaluate 100 random points
np.random.seed(42)
points = np.random.uniform(-5.0, 5.0, (100, dim))

# Evaluate the mathematically pure BBOB problem
clean_y = [noisy_func.clean_evaluate(x) for x in points]
noisy_y = [noisy_func.add_noise(y) for y in clean_y]

print(f"Mean clean fitness of sampled points: {np.mean(clean_y):.4f}")
print(f"Mean noisy fitness of sampled points: {np.mean(noisy_y):.4f}")

# Clear IOH evaluation history
noisy_func.reset()

# Plot the before (clean) and after (noisy) fitness landscape using Plotly
import plotly.graph_objects as go

# Sort points by their clean fitness to form a nice curve
sort_idx = np.argsort(clean_y)
sorted_clean = np.array(clean_y)[sort_idx]
sorted_noisy = np.array(noisy_y)[sort_idx]
indices = np.arange(len(points))

fig = go.Figure()

# 1. Clean fitness line (Before noise)
fig.add_trace(go.Scatter(
    x=indices,
    y=sorted_clean,
    mode='lines',
    name='Clean Fitness (Before Noise)',
    line=dict(color='#4F46E5', width=3)
))

# 2. Noisy fitness markers (After noise)
fig.add_trace(go.Scatter(
    x=indices,
    y=sorted_noisy,
    mode='markers',
    name='Noisy Fitness (After Noise)',
    marker=dict(color='#EF4444', size=6, opacity=0.75)
))

fig.update_layout(
    title=f"BBOB Problem {problem_id}: Before vs. After Noise Injection (Noise Std={noise_std})",
    xaxis_title="Sample Points (Sorted by Clean Fitness)",
    yaxis_title="Fitness Value",
    template="plotly_white",
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.8)'),
    margin=dict(l=40, r=40, t=60, b=40),
    hovermode='x unified'
)

fig.show()


Loaded Problem 1 (Instance 1, DIM=3)
True target optimum (minimum): 79.4800
Mean clean fitness of sampled points: 107.6693
Mean noisy fitness of sampled points: 107.7042


## 2. Inspecting Prompts
Let's verify the dynamic task prompt that gets passed to LLaMEA for problem evolution.

In [3]:
print("--- TASK PROMPT EXAMPLE (Problem 1) ---")
print(TASK_PROMPT_TEMPLATE.format(problem_id=1))
print("\n--- FORMAT PROMPT ---")
print(FORMAT_PROMPT)

--- TASK PROMPT EXAMPLE (Problem 1) ---

You are a highly skilled computer scientist and an expert in meta-heuristic optimization.
Your task is to design a novel, continuous black-box optimization algorithm that is highly specialized for a specific target landscape (BBOB Problem ID: 1).
Critically, the objective function you are optimizing contains statistical noise. Your algorithm must be resilient to this noise and avoid being trapped by false gradients.

This is NOT a general-purpose solver. You are designing a bespoke algorithm tailored to exploit the specific features of this single noisy landscape.

Write the Python code for a class that contains a `__call__(self, func, max_evaluations)` method.
The `func` is the noisy objective function to be minimized, taking a numpy array (the search point) and returning a float (the fitness).
The `max_evaluations` is the maximum number of times you can call `func`.
The domain bounds for the search space are [-5.0, 5.0] for all dimensions.
You

## 3. LLM Provider Configuration
Configure the API key, model name, and base URL endpoint using Option A configurations (conceptually identical to `main.py`).

In [4]:
import os
from dotenv import load_dotenv
from aad_llm.llm_providers import build_llm, Provider

# Load environment variables from .env if present
load_dotenv(dotenv_path="../../.env")

# Choose provider. Config values will be pulled automatically from env.
LLM_PROVIDER = os.environ.get("LLM_PROVIDER", Provider.GEMINI)

llm = build_llm(LLM_PROVIDER)

print("=================================================================")
print(f"Initialized Provider: {LLM_PROVIDER}")
print(f"Model Target:         {llm.model}")
print("=================================================================")

Initialized Provider: gemini
Model Target:         gemini-2.0-flash


## 4. Run Evolution on a Single Problem
Let's run a test evolution loop for 5 iterations on problems 1, 2, and 3 (max_evaluations 1000). 
Make sure your target server endpoint is active.

In [5]:
# Run the evolution loop synchronously
PROBLEMS = [1, 2, 3]
output_dir = "../../generated_algorithms"
log_dir = "../../logs"

results = runner.run_evolution_for_problems(
    problems=PROBLEMS,
    dim=3,
    noise_std=0.05,
    llm=llm,
    max_evaluations=1000,
    iterations=5,  # Running brief test runs
    output_dir=output_dir,
    log_dir=log_dir,
    verbose=True,
    log=False
)

print("\n======================================================")
print("Execution complete. Final results:")
for pid, score in results.items():
    score_str = f"{score:.4f}" if score is not None else "FAILED"
    print(f"  Problem {pid}: Score = {score_str}")
print("======================================================")

/Users/nicolaibrahim/Desktop/proj/AAD_LLM/.venv/lib/python3.12/site-packages/joblib/parallel.py:1383: UserWarning: The backend class 'SequentialBackend' does not support timeout. You have set 'timeout=3615' in Parallel but the 'timeout' parameter will not be used.
  warnings.warn(
2026-07-01 14:02:41 INFO AFC is enabled with max remote calls: 10.



>>> Evolving algorithm for BBOB Problem 1...
No archive path provided, standard initialisation.
Initializing first population


2026-07-01 14:02:41 INFO HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent "HTTP/1.1 429 Too Many Requests"
2026-07-01 14:02:51 INFO AFC is enabled with max remote calls: 10.
2026-07-01 14:02:51 INFO HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent "HTTP/1.1 429 Too Many Requests"


KeyboardInterrupt: 

## 5. Run the Full Experiment
To sequentially evolve optimization algorithms for all 24 BBOB functions, run the following block. Note that this requires access to the remote server and may take a significant amount of time.